# 05. Predict the future and distinguish two kinds of stochasticity

Random completion can use both temporal directions. This lesson trains a separate future-only branch, visualizes an approximate coordinate forecast, and explains why positional noise and latent samples answer different questions.

## Run locally

Run this notebook independently from the experiment directory. No previous notebook execution is required. The environment installs this local `worldmodels` package and the repository's editable Ambient package together.

```bash
cd experiments/aaai-worldmodels
uv sync
uv run python -m ipykernel install --user --name medwm-gait --display-name "Med-WM gait (Python 3.12)"
uv run jupyter lab
```

Select **Med-WM gait (Python 3.12)**. The project requires Python 3.12. Place local settings in this project's ignored `.env`. The requested `load_env(find_env())` spelling uses aliases for `python-dotenv`; it resolves this experiment's `.env` explicitly and never displays credentials. These lessons need no paid model API.

Set `WM_DATA_MODE=real` to use only actual prepared GAVD skeletons and real MediaPipe/PyTorch/classifier calls. Real mode requires the immutable person-gated clean cohort and fails if those inputs are absent; it never substitutes synthetic data. It embeds a short local walking excerpt and persists numerical outputs and trained lesson checkpoints under a separately named real execution. Run all real lessons with `uv run python scripts/execute_notebooks.py --data-mode real`.

The default `synthetic` mode is an optional fast teaching fixture. Its motions are not medical simulations and cannot establish clinical accuracy. Real lesson training is also a bounded tutorial exercise, distinct from the primary controlled study. Keep all three evidence types separate.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown, SVG
from worldmodels.settings import PROJECT, REPOSITORY, load_env, find_env
load_env(find_env())
torch.set_num_threads(2)
from worldmodels.training import seed_everything
from worldmodels.notebook_support import (data_mode, lesson_data, real_inputs, real_example,
    bounded_preview, lesson_directory, persist_lesson, prepared_path, comparison_path)
REAL = data_mode() == "real"
RELEASE_TEST_RESULTS = os.getenv("WM_RELEASE_TEST_RESULTS", "0") == "1"
EVIDENCE_LABEL = "Real GAVD" if REAL else "Synthetic teaching"
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
seed_everything(27)
print("Data mode:", data_mode(), "| Prepared source:", prepared_path() if REAL else "synthetic fixture")
print("Project:", PROJECT.name, "| Python kernel ready")

### Walking-video context

Real mode embeds only a short original walking excerpt from an accepted normal training sequence. The preview is saved under `data/videos-gavd/previews/`, with its source hash and frame range. It provides context for this lesson without duplicating the full original. Synthetic mode enables the same real viewing aid only when `WM_EMBED_VIDEO=1`.

In [ ]:
if REAL or os.getenv("WM_EMBED_VIDEO", "0") == "1":
    from IPython.display import Video
    preview_path = bounded_preview()
    display(Video(str(preview_path), embed=True, html_attributes='controls preload="metadata"'))
    print("Bounded source excerpt:", preview_path.relative_to(REPOSITORY))
else:
    print("Synthetic mode: set WM_EMBED_VIDEO=1 for a real-video context excerpt.")

## Step 1. Make the future inaccessible to the predictor

The first eight frames are context; the last eight are targets. All future joints are hidden from the online encoder. The teacher may observe the outcome when computing a training target or retrospective residual. It must not feed those outcomes into deployment predictions. Prefix-only normalization avoids using future scale statistics, while future pelvis centering defines an articulation target rather than global translation.

In [ ]:
display(SVG(filename=str(PROJECT / "images/forecast.svg")))

In [ ]:
import copy
from worldmodels.notebook_support import lesson_data
from worldmodels.model import ModelConfig, SkeletonJEPA
from worldmodels.training import TrainConfig, fit, anomaly_scores
fixture = lesson_data(n=16, seed=27)
def train_stage(net, name, regime, forecast_fraction=0.0):
    return fit(net, fixture["x"], fixture["valid"],
        TrainConfig(steps=4, seed=27, regime=regime, run_id=name, forecast_fraction=forecast_fraction, latent_mse_weight=1.0),
        labels=fixture["labels"], source_ids=fixture["source_ids"], partitions=["train"]*len(fixture["x"]))
normal = SkeletonJEPA(ModelConfig(dim=24, encoder_layers=1))
normal_initial_state = copy.deepcopy(normal.state_dict())
deterministic_history = {"normal": train_stage(normal, "lesson05-normal-v1", "normal")}
forecast = copy.deepcopy(normal)
deterministic_history["forecast"] = train_stage(forecast, "lesson05-forecast-v1", "forecast", 1.0)

## Step 2. Inspect one coordinate forecast

The coordinate readout is trained alongside the latent objective. A tiny run is expected to produce rough trajectories. Real mode plots actual GAVD articulation; synthetic mode plots a software fixture. In either mode this is a diagnostic for units, horizon, and target visibility, rather than clinical validation. A useful future model must beat simple controls such as repeating the last observed articulation.

In [ ]:
evaluation_fixture = lesson_data(n=8, seed=27, split="validation") if REAL else fixture
scores = anomaly_scores(forecast, evaluation_fixture["x"][:4], evaluation_fixture["valid"][:4], context_frames=8)
joint_slot = 10  # left ankle, BlazePose 27
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(np.arange(16), evaluation_fixture["x"][0,:,joint_slot,0], label=EVIDENCE_LABEL + " observed trajectory")
ax.plot(np.arange(8,16), scores["coordinates"][0,8:,joint_slot,0], label="Predicted future")
ax.axvline(7.5,color="gray",linestyle=":")
ax.set(xlabel="Frame slot",ylabel="Relative x",title=EVIDENCE_LABEL + " left-ankle forecast")
ax.legend(); plt.show()

## Step 3. Keep a persistence baseline

Persistence predicts that the relative articulation stays at its last observed value. It can be surprisingly strong at short horizons. Compute its error on exactly the same target mask and coordinate units. If a learned model cannot improve this reference, claims of useful dynamics learning need revision.

In [ ]:
target = scores["target_mask"]
persistence = np.repeat(evaluation_fixture["x"][:4,7:8], 16, axis=1)
squared = ((persistence-evaluation_fixture["x"][:4])**2).mean(-1)
persistence_error = np.array([squared[i][target[i]].mean() for i in range(4)])
display(pd.DataFrame({"learned_coordinate_mse": scores["coordinate_mse"], "persistence_coordinate_mse": persistence_error}))

## Step 4. Distinguish positional noise from latent possibilities

Positional noise perturbs where a token is represented. The implemented temporal jitter is a proposed ablation and is not a faithful implementation of the published StoP weight-tying mechanism. Anatomical joint identities remain fixed.

A stochastic latent variable represents alternative completions conditioned on the same visible past. Its posterior can use targets during training, while its deployment prior uses context only. The KL term limits the gap between these distributions. It does not replace anti-collapse regularization. The recent Stochastic-JEPA image results do not establish a benefit for gait. This lesson gives both deterministic and stochastic branches the same latent-MSE weight of 1.0 alongside categorical teacher loss, coordinate readout, and VICReg terms. The stochastic branch additionally uses KL regularization. This is a declared hybrid adaptation, not a reproduction of the published latent-MSE-plus-KL objective alone.

In [ ]:
display(SVG(filename=str(PROJECT / "images/stochasticity.svg")))

### Compare temporal-position jitter on the same real inputs

The next branch uses temporal jitter with standard deviation 0.2 in frame-slot units. It starts from exactly the deterministic branch's initial weights, uses the same source batches, uniform-mask seed, hybrid objective, and four updates in each normal and forecast stage. The implementation perturbs shared temporal encodings during training; it preserves joint identities and uses exact positions at evaluation. This is a simple positional-stochasticity experiment, not the published StoP weight-tying mechanism.

Real mode trains both branches on actual normal GAVD training sources and evaluates the same normal validation windows. The tiny, one-seed table demonstrates a controlled software comparison. It does not establish that positional noise improves performance and is separate from the main three-seed revision study.

In [ ]:
positional = SkeletonJEPA(ModelConfig(dim=24, encoder_layers=1, position_jitter=0.2, latent_dim=0))
positional.load_state_dict(normal_initial_state)
assert all(torch.equal(normal_initial_state[key], positional.state_dict()[key]) for key in normal_initial_state)
positional_history = {"normal": train_stage(positional, "lesson05-position-normal-v1", "normal")}
positional_history["forecast"] = train_stage(positional, "lesson05-position-forecast-v1", "forecast", 1.0)
assert positional.run_metadata["data_sha256"] == forecast.run_metadata["data_sha256"]
assert all(value == forecast.run_metadata["config"][key] for key, value in positional.run_metadata["config"].items() if key != "run_id")
positional_scores = anomaly_scores(positional, evaluation_fixture["x"][:4], evaluation_fixture["valid"][:4], context_frames=8)
positional_comparison = pd.DataFrame({
    "source_id": evaluation_fixture["source_ids"][:4],
    "deterministic_mse": scores["coordinate_mse"],
    "position_jitter_mse": positional_scores["coordinate_mse"],
    "persistence_mse": persistence_error,
})
display(positional_comparison)
print("Matched initial weights and batches; four updates per stage; one teaching seed.")

In [ ]:
seed_everything(27)
stochastic = SkeletonJEPA(ModelConfig(dim=24, encoder_layers=1, latent_dim=4))
stochastic.load_state_dict(normal_initial_state, strict=False)
stochastic_history = {"normal": train_stage(stochastic, "lesson05-latent-normal-v2", "normal")}
stochastic_history["forecast"] = train_stage(stochastic, "lesson05-latent-forecast-v2", "forecast", 1.0)
sampled = anomaly_scores(stochastic, evaluation_fixture["x"][:4], evaluation_fixture["valid"][:4], context_frames=8, samples=8, seed=27)
print("Prior sample tensor:", sampled["samples"].shape)
display(pd.DataFrame({"prediction_error": sampled["coordinate_mse"], "uncalibrated_dispersion": sampled["predictive_variance"]}))

## Step 5. Do not call dispersion calibrated uncertainty

Multiple prior samples can differ without their spread tracking predictive error. Test error versus dispersion on held-out sources, assess risk when uncertain cases are deferred, and compare uncertainty with pose visibility and camera view. High spread may reflect acquisition noise rather than disease. A few teaching examples cannot establish calibration, even when they come from real GAVD recordings.

For a normal-reference detector, choose a threshold on normal validation sources only. With six such sources, the conservative finite-sample threshold for a 5% false-positive target is infinite. This means there is not enough calibration support for a useful finite guarantee.

In [ ]:
from worldmodels.evaluation import validation_threshold_report
if REAL:
    from worldmodels.evaluation import aggregate_sources
    validation_fixture = lesson_data(n=32, conditions=False, split="validation")
    validation_scores = anomaly_scores(forecast, validation_fixture["x"], validation_fixture["valid"], context_frames=8)["coordinate_mse"]
    _, _, source_calibration = aggregate_sources(validation_scores[:,None], validation_fixture["labels"], validation_fixture["source_ids"], validation_fixture["metadata"].sequence_id)
    calibration_example = validation_threshold_report(source_calibration[:,0])
else:
    calibration_example = validation_threshold_report([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
display(pd.Series(calibration_example))

## Step 6. Bound explanations

A large left-ankle residual identifies where the prediction disagreed with an observed estimate. It does not identify the neurological cause. Check occlusion, tracking failures, camera direction, and unusual action before interpreting a residual as abnormal gait. The normal-only reference and adapted classifier should keep separate names and evidence records.

In [ ]:
joint_residual = ((scores["coordinates"][0]-evaluation_fixture["x"][0])**2).mean(-1)
fig, ax = plt.subplots(figsize=(9,3))
ax.imshow(joint_residual[8:].T, aspect="auto", cmap="magma")
ax.set(xlabel="Future frame slot",ylabel="Joint slot",title=EVIDENCE_LABEL + " residual localization, not causal explanation")
plt.show()

In [ ]:
lesson_evidence = persist_lesson("05_forecasting_and_stochasticity", globals())
print("Evidence mode:", lesson_evidence.get("mode"))

## What to retain

Record the data identity, settings, executed commands, and limitations with any result. A successful cell execution is implementation evidence, not proof that the method generalizes. See the [data protocol](docs/data_protocol.md), [method review](docs/literature.md), and [README](README.md) before changing the study.